In [1]:
!pip install opencv-python-headless keras tensorflow dlib imutils

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import zipfile

# Specify the path to the dataset zip file in Google Drive
dataset_zip_path = '/content/drive/MyDrive/Colab Notebooks/anjana/Driver Drowsiness Dataset.zip'

# Extract the zip file to a directory
extract_dir = '/content/drive/MyDrive/Colab Notebooks/anjana/dataset' # Choose an appropriate directory
with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Now you can list the contents of the extracted directory
os.listdir(extract_dir)

['Driver Drowsiness Dataset (DDD)']

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator # Update import statement

# Initialize the ImageDataGenerator
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# Prepare iterators
train_generator = datagen.flow_from_directory(
    extract_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    extract_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 33435 images belonging to 1 classes.
Found 8358 images belonging to 1 classes.


In [10]:



from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=1

)


1044/1044 ━━━━━━━━━━━━━━━━━━━━ 550s 524ms/step - accuracy: 0.9982 - loss: 0.0043 - val_accuracy: 1.0000 - val_loss: 4.6010e-20


In [12]:
# Save the model in HDF5 format
model.save('saved_model/my_model.h5')

In [13]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 20.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-metadata 1.15.0 requires protobuf<4.21,>=3.20.3; python_version < "3.11", but you have protobuf 4.25.4 which is incompatible.


In [1]:
!pip install opencv-python-headless mediapipe tensorflow


In [2]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

def capture_image(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function captureImage(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Resize the output to match the video resolution
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks()[0].stop();
            div.remove();

            const dataUrl = canvas.toDataURL('image/jpeg', quality);
            return dataUrl;
        }
        ''')
    display(js)
    data = eval_js('captureImage({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

# Test capturing a photo
try:
    filename = capture_image()
    print('Saved to {}'.format(filename))

    # Display the captured image
    img = cv2.imread(filename)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    from matplotlib import pyplot as plt
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()
except Exception as e:
    print(str(e))


<IPython.core.display.Javascript object>

NotAllowedError: Permission denied


Real-Time Detection: Capture video frames, detect facial landmarks using MediaPipe, predict eye states, and detect blinks, yawns, and head pose.

Blink Detection: Count frames where EAR is below the threshold to detect blinks.

Yawn Detection: Count frames where MAR is above the threshold to detect yawns.

 Head Pose Estimation: Use a pose estimation algorithm to determine the head orientation.

In [3]:
import cv2
import numpy as np
import mediapipe as mp
from scipy.spatial import distance as dist
from tensorflow.keras.models import load_model

# Load the trained eye state model
model = load_model('saved_model/my_model.h5', compile=False)
# Compile the model with appropriate loss and metrics
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Define function for eye aspect ratio calculation
def eye_aspect_ratio(eye):
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    C = dist.euclidean(eye[0], eye[3])
    ear = (A + B) / (2.0 * C)
    return ear

# Define function for mouth aspect ratio calculation
def mouth_aspect_ratio(mouth):
    A = dist.euclidean(mouth[2], mouth[10])
    B = dist.euclidean(mouth[4], mouth[8])
    C = dist.euclidean(mouth[0], mouth[6])
    mar = (A + B) / (2.0 * C)
    return mar

# Define function to preprocess input for model prediction
def preprocess_input(eye_image):
    gray = cv2.cvtColor(eye_image, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (24, 24))
    normalized = resized / 255.0
    reshaped = np.reshape(normalized, (1, 24, 24, 1))
    return reshaped

# Process the captured image for drowsiness detection
img = cv2.imread(filename)
frame = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

results = face_mesh.process(frame)
if results.multi_face_landmarks:
    for face_landmarks in results.multi_face_landmarks:
        landmarks = [(int(l.x * frame.shape[1]), int(l.y * frame.shape[0])) for l in face_landmarks.landmark]

        # Left eye landmarks (right eye for the viewer)
        left_eye_indices = [362, 385, 387, 263, 373, 380]
        left_eye = np.array([landmarks[idx] for idx in left_eye_indices])

        # Right eye landmarks (left eye for the viewer)
        right_eye_indices = [33, 160, 158, 133, 153, 144]
        right_eye = np.array([landmarks[idx] for idx in right_eye_indices])

        leftEAR = eye_aspect_ratio(left_eye)
        rightEAR = eye_aspect_ratio(right_eye)
        ear = (leftEAR + rightEAR) / 2.0

        if ear < BLINK_THRESHOLD:
            blink_counter += 1
        else:
            if blink_counter >= BLINK_CONSEC_FRAMES:
                cv2.putText(frame, "Blink", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            blink_counter = 0

        for eye, name in zip([left_eye, right_eye], ["Left Eye", "Right Eye"]):
            x_min, y_min = np.min(eye, axis=0)
            x_max, y_max = np.max(eye, axis=0)

            eye_image = frame[y_min:y_max, x_min:x_max]
            if eye_image.size > 0:
                eye_image = preprocess_input(eye_image)
                prediction = model.predict(eye_image)
                if prediction[0][0] > 0.5:
                    cv2.putText(frame, f'{name} Open', (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                else:
                    cv2.putText(frame, f'{name} Closed', (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        # Mouth landmarks
        mouth_indices = [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308, 191]
        mouth = np.array([landmarks[idx] for idx in mouth_indices])

        mar = mouth_aspect_ratio(mouth)

        if mar > YAWN_THRESHOLD:
            yawn_counter += 1
        else:
            if yawn_counter >= YAWN_CONSEC_FRAMES:
                cv2.putText(frame, "Yawn", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            yawn_counter = 0

# Display the processed frame
plt.imshow(frame)
plt.axis('off')
plt.show()


NameError: name 'filename' is not defined